In [1]:
from tensorflow.keras.models import load_model
from qkeras.utils import _add_supported_quantized_objects
import os

co = {}
_add_supported_quantized_objects(co)

os.environ['PATH'] = os.environ['XILINX_VITIS'] + '/bin:' + os.environ['PATH']
os.environ['PATH'] = os.environ['XILINX_VIVADO'] + '/bin:' + os.environ['PATH']

2026-07-31 00:57:39.278163: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-07-31 00:57:39.302611: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-07-31 00:57:39.302637: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-07-31 00:57:39.303400: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-07-31 00:57:39.307508: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-07-31 00:57:39.307886: I tensorflow/core/platform/cpu_feature_guard.cc:1

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

2026-07-31 00:57:40.072929: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [2]:
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Activation, BatchNormalization, Dropout
from tensorflow.keras.regularizers import l1
from tensorflow.keras.initializers import LecunUniform
from qkeras.qlayers import QDense, QActivation
from qkeras.quantizers import quantized_bits, quantized_relu
import tensorflow_model_optimization as tfmot
from tensorflow_model_optimization.sparsity.keras import strip_pruning

# Load the test data
X_test = np.ascontiguousarray(np.load('X_test.npy'))
y_test = np.load('y_test.npy', allow_pickle=True)
classes = np.load('classes.npy', allow_pickle=True)

# Manually rebuild the QKeras architecture
model = Sequential()
model.add(QDense(64, input_shape=(8,), name='fc1',
                 kernel_quantizer=quantized_bits(6, 2, alpha=1),
                 bias_quantizer=quantized_bits(6, 2, alpha=1),
                 kernel_initializer=LecunUniform(), kernel_regularizer=l1(0.001)))
model.add(BatchNormalization())
model.add(QActivation(activation=quantized_relu(6, 2), name='relu1'))
model.add(Dropout(0.2))
model.add(QDense(32, name='fc2',
                 kernel_quantizer=quantized_bits(6, 2, alpha=1),
                 bias_quantizer=quantized_bits(6, 2, alpha=1),
                 kernel_initializer=LecunUniform(), kernel_regularizer=l1(0.001)))
model.add(QActivation(activation=quantized_relu(6, 2), name='relu2'))
model.add(Dropout(0.2))
model.add(QDense(1, name='output',
                 kernel_quantizer=quantized_bits(6, 2, alpha=1),
                 bias_quantizer=quantized_bits(6, 2, alpha=1),
                 kernel_initializer=LecunUniform(), kernel_regularizer=l1(0.001)))
model.add(Activation('sigmoid', name='sigmoid'))

# Apply the pruning wrapper so the structure perfectly matches the checkpoint
pruning_params = {
    'pruning_schedule': tfmot.sparsity.keras.ConstantSparsity(0.5, begin_step=0, frequency=100)
}
model = tfmot.sparsity.keras.prune_low_magnitude(model, **pruning_params)

# Load only the weights
model.load_weights('model_3/KERAS_check_best_model_weights.h5')

# Strip the pruning wrappers to get a clean QKeras model ready for hls4ml
model = strip_pruning(model)

print("Successfully loaded weights and stripped pruning wrappers!")

Successfully loaded weights and stripped pruning wrappers!


In [3]:
import hls4ml
import plotting

config = hls4ml.utils.config_from_keras_model(model, granularity='name')
config['LayerName']['sigmoid']['exp_table_t'] = 'ap_fixed<18,8>'
config['LayerName']['sigmoid']['inv_table_t'] = 'ap_fixed<18,4>'
config['ReuseFactor'] = 16
print("-----------------------------------")
plotting.print_dict(config)
print("-----------------------------------")
hls_model = hls4ml.converters.convert_from_keras_model(
    model, 
    hls_config=config, 
    backend='Vitis', 
    output_dir='model_3/hls4ml_prj_alveo', 
    part='xcu250-figd2104-2L-e'
)
hls_model.compile()

/home/tvangos/.local/lib/python3.10/site-packages/keras/src/constraints.py:365: UserWarning: The `keras.constraints.serialize()` API should only be used for objects of type `keras.constraints.Constraint`. Found an instance of type <class 'qkeras.quantizers.quantized_bits'>, which may lead to improper serialization.
  warnings.warn(


-----------------------------------
Model
  Precision
    default:         fixed<16,6>
  ReuseFactor:       1
  Strategy:          Latency
  BramFactor:        1000000000
  TraceOutput:       False
LayerName
  fc1_input
    Trace:           False
    Precision
      result:        auto
  fc1
    Trace:           False
    Precision
      result:        auto
      weight:        fixed<6,3,TRN,WRAP,0>
      bias:          fixed<6,3,TRN,WRAP,0>
  fc1_linear
    Trace:           False
    Precision
      result:        auto
  batch_normalization
    Trace:           False
    Precision
      result:        auto
      scale:         auto
      bias:          auto
  relu1
    Trace:           False
    Precision
      result:        ufixed<6,2,RND_CONV,SAT,0>
  fc2
    Trace:           False
    Precision
      result:        auto
      weight:        fixed<6,3,TRN,WRAP,0>
      bias:          fixed<6,3,TRN,WRAP,0>
  fc2_linear
    Trace:           False
    Precision
      result:        au

In [4]:
# Print the Vitis backend config instead
plotting.print_dict(hls4ml.backends.get_backend('Vitis').create_initial_config())

Part:                xcvu13p-flga2577-2-e
ClockPeriod:         5
ClockUncertainty:    27%
IOType:              io_parallel
HLSConfig
WriterConfig
  Namespace:         None
  WriteWeightsTxt:   True
  WriteTar:          False
  TBOutputStream:    both
  WriteEmulationConstants:False


In [5]:
import numpy as np

X_test = np.load('X_test.npy')
y_hls = hls_model.predict(np.ascontiguousarray(X_test))
np.save('model_3/y_hls.npy', y_hls)

In [6]:
# synth=True runs C-synthesis
# vsynth=True runs Vivado Logic Synthesis
hls_model.build(csim=False, synth=True, vsynth=True, export=True)


****** vitis-run v2023.2 (64-bit)
  **** SW Build 4026344 on 2023-10-11-15:42:10
    ** Copyright 1986-2022 Xilinx, Inc. All Rights Reserved.
    ** Copyright 2022-2023 Advanced Micro Devices, Inc. All Rights Reserved.

INFO: [vitis-run 82-31] Launching vitis_hls: vitis_hls -nolog -run tcl -f /home/tvangos/Desktop/UTH/CPU_design/src/model_3/hls4ml_prj_alveo/build_prj.tcl -work_dir /home/tvangos/Desktop/UTH/CPU_design/src/model_3/hls4ml_prj_alveo

****** Vitis HLS - High-Level Synthesis from C, C++ and OpenCL v2023.2 (64-bit)
  **** SW Build 4023990 on Oct 11 2023
  **** IP Build 4028589 on Sat Oct 14 00:45:43 MDT 2023
  **** SharedData Build 4025554 on Tue Oct 10 17:18:54 MDT 2023
    ** Copyright 1986-2022 Xilinx, Inc. All Rights Reserved.
    ** Copyright 2022-2023 Advanced Micro Devices, Inc. All Rights Reserved.

source /tools/Xilinx/Vitis_HLS/2023.2/scripts/vitis_hls/hls.tcl -notrace
INFO: [HLS 200-10] Running '/tools/Xilinx/Vitis_HLS/2023.2/bin/unwrapped/lnx64.o/vitis_hls'
INFO:

{'CSynthesisReport': {'TargetClockPeriod': '5.00',
  'EstimatedClockPeriod': '3.646',
  'BestLatency': '9',
  'WorstLatency': '9',
  'IntervalMin': '1',
  'IntervalMax': '1',
  'BRAM_18K': '1',
  'DSP': '62',
  'FF': '8376',
  'LUT': '33655',
  'URAM': '0',
  'AvailableBRAM_18K': '5376',
  'AvailableDSP': '12288',
  'AvailableFF': '3456000',
  'AvailableLUT': '1728000',
  'AvailableURAM': '1280'},
 'VivadoSynthReport': {'LUT': '10163',
  'FF': '3828',
  'BRAM_18K': '0.5',
  'URAM': '0',
  'DSP48E': '62'}}

In [7]:
import os
import glob

output_dir = 'model_3/hls4ml_prj_alveo' 

# Find all .rpt files generated by Vivado in the output folder
report_files = glob.glob(f'{output_dir}/**/*.rpt', recursive=True)

for file in report_files:
    # We specifically want to look at the synthesis power and timing reports
    if 'power' in file.lower() or 'timing' in file.lower() or 'vivado_synth' in file.lower():
        print("="*80)
        print(f"REPORT FILE: {file}")
        print("="*80)
        with open(file, 'r') as f:
            # Print the first 100 lines
            lines = f.readlines()
            print("".join(lines[:100])) 
        print("\n\n")

REPORT FILE: model_3/hls4ml_prj_alveo/power_synth.rpt
Copyright 1986-2022 Xilinx, Inc. All Rights Reserved. Copyright 2022-2023 Advanced Micro Devices, Inc. All Rights Reserved.
-------------------------------------------------------------------------------------------------------------------------------------------------
| Tool Version     : Vivado v.2023.2 (lin64) Build 4029153 Fri Oct 13 20:13:54 MDT 2023
| Date             : Thu Jul 30 02:37:52 2026
| Host             : tvangos-MS-7C75 running 64-bit Ubuntu 22.04.5 LTS
| Command          : report_power -file power_synth.rpt
| Design           : myproject
| Device           : xc7z020clg484-1
| Design State     : synthesized
| Grade            : commercial
| Process          : typical
| Characterization : Production
-------------------------------------------------------------------------------------------------------------------------------------------------

Power Report

Table of Contents
-----------------
1. Summary
1.1 On-Chip C

In [8]:
import os
import glob

output_dir = 'model_3/hls4ml_prj_alveo'

# Find the vivado synthesis script generated by hls4ml
tcl_files = glob.glob(f'{output_dir}/**/vivado_synth.tcl', recursive=True)

if not tcl_files:
    print("Could not find vivado_synth.tcl. Check your output_dir path!")
else:
    tcl_file = tcl_files[0]
    print(f"Found {tcl_file}! Injecting report commands...")

    # Append the missing report commands to the end of the script
    with open(tcl_file, 'a') as f:
        f.write("\n# --- Added manually to generate missing reports ---\n")
        f.write("report_timing_summary -file timing_summary_synth.rpt\n")
        f.write("report_power -file power_synth.rpt\n")

    # Add vivado to PATH 
    vivado_path = '/tools/Xilinx/Vivado/2023.2/bin'
    os.environ['PATH'] = vivado_path + ':' + os.environ['PATH']

    # Run vivado synthesis again
    tcl_dir = os.path.dirname(tcl_file)
    print(f"Running Vivado in {tcl_dir} (This will take a few minutes)...")
    
    # Execute Vivado in batch mode using the modified script
    os.system(f"cd {tcl_dir} && vivado -mode batch -source vivado_synth.tcl")
    
    print("Finished! The reports should now be generated.")

Found model_3/hls4ml_prj_alveo/vivado_synth.tcl! Injecting report commands...
Running Vivado in model_3/hls4ml_prj_alveo (This will take a few minutes)...

****** Vivado v2023.2 (64-bit)
  **** SW Build 4029153 on Fri Oct 13 20:13:54 MDT 2023
  **** IP Build 4028589 on Sat Oct 14 00:45:43 MDT 2023
  **** SharedData Build 4025554 on Tue Oct 10 17:18:54 MDT 2023
    ** Copyright 1986-2022 Xilinx, Inc. All Rights Reserved.
    ** Copyright 2022-2023 Advanced Micro Devices, Inc. All Rights Reserved.

source vivado_synth.tcl
# set tcldir [file dirname [info script]]
# source [file join $tcldir project.tcl]
## variable project_name
## set project_name "myproject"
## variable backend
## set backend "vitis"
## variable part
## set part "xcu250-figd2104-2L-e"
## variable clock_period
## set clock_period 5
## variable clock_uncertainty
## set clock_uncertainty 27%
## variable version
## set version "1.0.0"
## variable maximum_size
## set maximum_size 4096
# add_files ${project_name}_prj/solution

In [10]:
!head -n 50 model_3/hls4ml_prj_alveo/myproject_prj/solution1/syn/report/myproject_csynth.rpt



== Vitis HLS Report for 'myproject'
* Date:           Fri Jul 31 00:58:37 2026

* Version:        2023.2 (Build 4023990 on Oct 11 2023)
* Project:        myproject_prj
* Solution:       solution1 (Vivado IP Flow Target)
* Product family: virtexuplus
* Target device:  xcu250-figd2104-2L-e


== Performance Estimates
+ Timing: 
    * Summary: 
    +--------+---------+----------+------------+
    |  Clock |  Target | Estimated| Uncertainty|
    +--------+---------+----------+------------+
    |ap_clk  |  5.00 ns|  3.646 ns|     1.35 ns|
    +--------+---------+----------+------------+

+ Latency: 
    * Summary: 
    +---------+---------+-----------+-----------+-----+-----+---------+
    |  Latency (cycles) |   Latency (absolute)  |  Interval | Pipeline|
    |   min   |   max   |    min    |    max    | min | max |   Type  |
    +---------+---------+-----------+-----------+-----+-----+---------+
    |        9|        9|  45.000 ns|  45.000 ns|    1|    1|      yes|
    +---------+------

In [11]:
!mkdir -p model_3/hls4ml_prj_alveo/package
# Copy the generated IP zip file instead of a bitstream
!cp model_3/hls4ml_prj_alveo/myproject_prj/solution1/impl/ip/*.zip model_3/hls4ml_prj_alveo/package/hls4ml_nn_ip.zip
!cp X_test.npy y_test.npy model_3/hls4ml_prj_alveo/package
!cp part7b_deployment.ipynb model_3/hls4ml_prj_alveo/package
!tar -czvf model_3/hls4ml_prj_alveo/package.tar.gz -C model_3/hls4ml_prj_alveo/package/ .

cp: cannot stat 'part7b_deployment.ipynb': No such file or directory
./
./y_test.npy
./X_test.npy
./hls4ml_nn_ip.zip
